(dem_manager)=
# DEM Manager: Fourteen Selectable DEM Products

`DEMManager` turns a geographic extent into a ready-to-use DEM without
manual downloading: it queries the raw tiles that cover the extent, reuses
tiles already present in a local cache, downloads the missing ones in
parallel (concurrent tiles plus ranged-chunk transfer), and mosaics them
into a single GeoTIFF.

Since PROPOSAL-0030 the manager is a **two-axis** contract:

- the **product** axis — what the data is: Copernicus GLO-30/GLO-90,
  NASADEM, ALOS World 3D, SRTM, terrain-tiles, ArcticDEM/REMA tiers,
  NISAR DEM — fourteen selection names in total (including `auto`);
- the **provider** axis — where it is fetched from: AWS, Planetary
  Computer, Earthdata, JAXA FTP.

Defaults are always **anonymous cloud channels** (AWS or Planetary
Computer): no account, no token, no API key. Switching provider is manual
and explicit — the manager never silently switches product or provider,
and `auto` never crosses the provider axis. When a provider fails you get
a structured `DEMProviderUnavailableError` instead of a hidden fallback.

This tutorial covers:

1. The fourteen selection names and their metadata
2. Building a combined DEM from cached tiles (default `glo30`)
3. Switching products and reading vertical datums
4. Switching providers manually
5. What `auto` does (and does not do)
6. Letting `run_pair` resolve the DEM automatically
7. Outage handling and environment variables

> **Prerequisites.** This notebook assumes you are running inside the
> project's `.venv` with all dependencies installed. `DEMManager` needs a
> cache folder; if `FANINSAR_DEM_CACHE_DIR` is already set in your
> environment it is reused, otherwise a temporary folder is created.
> Planetary Computer products additionally need the `[pc]` extra
> (`pip install "faninsar[pc]"`).

## Imports and cache setup


In [1]:
import os
import tempfile
from pathlib import Path

from faninsar.processing.geometry import DEMManager, get_dem_manager
from faninsar.processing.geometry.dem_sources import (
    list_dem_sources,
    parse_selection,
)
from faninsar.query import BoundingBox

cache_dir = Path(os.environ.get("FANINSAR_DEM_CACHE_DIR") or tempfile.mkdtemp(prefix="dem_cache_"))
os.environ["FANINSAR_DEM_CACHE_DIR"] = str(cache_dir)
print("tile cache:", cache_dir)

manager = get_dem_manager()  # no argument -> FANINSAR_DEM_SOURCE -> glo30
print("default selection:", manager.source_entry.name)

tile cache: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_cache_qsvf613p
default selection: glo30


## 1. The fourteen selection names

`list_dem_sources()` returns every registered name. The selection grammar
is `"<product>"` or `"<product>:<provider>"`; `"auto"` is the only
non-product name.


In [2]:
names = list_dem_sources()
print(f"{len(names)} selection names:")
print(", ".join(names))

# Resolve every bare product name and show the two-axis metadata.
# ("auto" is a manager-level alias, not a registry product; see section 5.)
header = f"{'selection':<15} {'provider':<11} {'resolution':>12}  {'vertical datum':<14} auth"
print("\n" + header + "\n" + "-" * len(header))
for name in names:
    if name == "auto":
        print(f"{name:<15} {'aws':<11} {'30 m':>12}  {'egm2008':<14} none  (GLO-30 + same-family GLO-90 rescue)")
        continue
    entry = parse_selection(name)
    resolution = f"{entry.resolution_m * 3600:.1f}\"" if entry.resolution_m < 1 else f"{entry.resolution_m:.0f} m"
    print(f"{name:<15} {entry.provider:<11} {resolution:>12}  {entry.vertical_datum:<14} {entry.auth}")

14 selection names:
alos-dem, arcticdem-10, arcticdem-2, arcticdem-32, auto, glo30, glo90, nasadem, nisar-glo30, rema-10, rema-2, rema-32, srtm-skadi, terrain-tiles

selection       provider      resolution  vertical datum auth
-------------------------------------------------------------
alos-dem        pc                  1.0"  egm96          none
arcticdem-10    aws                 30 m  ellipsoidal    none
arcticdem-2     aws                  2 m  ellipsoidal    none
arcticdem-32    aws                 32 m  ellipsoidal    none
auto            aws                 30 m  egm2008        none  (GLO-30 + same-family GLO-90 rescue)
glo30           aws                 1.0"  egm2008        none
glo90           aws                 3.0"  egm2008        none
nasadem         pc                  1.0"  egm96          none
nisar-glo30     earthdata           1.0"  ellipsoidal    token
rema-10         aws                 30 m  ellipsoidal    none
rema-2          aws                  2 m  ellipsoid

Two columns deserve attention:

- **vertical datum** — `egm2008`/`egm96` are orthometric (geoid) heights;
  `ellipsoidal` sources (`nisar-glo30`, `arcticdem-*`, `rema-*`) are
  already referenced to the WGS84 ellipsoid and must **not** be wrapped
  with a geoid correction again. `DEMManager.vertical_datum` reports the
  live value, and the pipeline entry point `resolve_auto_dem` applies the
  wrap rule automatically.
- **auth** — `none` works out of the box. `token` providers need
  Earthdata credentials (see section 4).


## 2. Build a combined DEM

`fetch_dem(bounds, output_path)` ensures every required tile is cached,
downloads the missing ones, and mosaics them into one file. Bounds are
`(min_lon, min_lat, max_lon, max_lat)`.


In [3]:
roi = BoundingBox(99.5, 38.5, 100.5, 39.5)
tiles = manager.required_tiles(roi)
stems = sorted(Path(t.cache_path).name for t in tiles)
print(f"{len(tiles)} tiles cover the ROI: {stems[:2]} ...")

out_dir = Path(tempfile.mkdtemp(prefix="dem_out_"))
dem_path = manager.fetch_dem(roi, out_dir / "dem.tif")
print("mosaic:", dem_path)

4 tiles cover the ROI: ['Copernicus_DSM_COG_10_N38_00_E099_00_DEM.tif', 'Copernicus_DSM_COG_10_N38_00_E100_00_DEM.tif'] ...


2026-08-24 21:11:43 | DEBUG | faninsar.processing.geometry.dem_transport | HEAD probe failed for https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N38_00_E100_00_DEM/Copernicus_DSM_COG_10_N38_00_E100_00_DEM.tif: SSLError


2026-08-24 21:11:43 | DEBUG | faninsar.processing.geometry.dem_transport | HEAD probe failed for https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N38_00_E099_00_DEM/Copernicus_DSM_COG_10_N38_00_E099_00_DEM.tif: SSLError


2026-08-24 21:11:43 | DEBUG | faninsar.processing.geometry.dem_transport | HEAD probe failed for https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N39_00_E099_00_DEM/Copernicus_DSM_COG_10_N39_00_E099_00_DEM.tif: SSLError


2026-08-24 21:11:43 | DEBUG | faninsar.processing.geometry.dem_transport | HEAD probe failed for https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_N39_00_E100_00_DEM/Copernicus_DSM_COG_10_N39_00_E100_00_DEM.tif: SSLError


...

2026-08-24 21:13:05 | DEBUG | faninsar.processing.geometry.dem_transport | tile sha256 Copernicus_DSM_COG_10_N38_00_E099_00_DEM.tif: eaf17a19280339b2170f56a53a44b579029e7f4e01ce6cce6e4734776523ad45


2026-08-24 21:13:09 | DEBUG | faninsar.processing.geometry.dem_transport | tile sha256 Copernicus_DSM_COG_10_N38_00_E100_00_DEM.tif: 08f2349f4f007be844dc594d3d2f42e8604102635d24362748ae4029e1384b73


2026-08-24 21:13:15 | DEBUG | faninsar.processing.geometry.dem_transport | tile sha256 Copernicus_DSM_COG_10_N39_00_E099_00_DEM.tif: 41e31d8e2aa3eea1dbb973d8216fc6dbcda24e1190afed53e629259b2e8f8a0d


2026-08-24 21:13:20 | INFO | faninsar.processing.geometry.dem_manager | DEM mosaic written: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_out_2pvgqaxl/dem.tif shape=(7200, 7200) source=glo30@aws


mosaic: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/dem_out_2pvgqaxl/dem.tif


Tiles land under `<cache_dir>/<product>-<provider>/`, so switching
selections never invalidates other channels' tiles; a second call with a
warm cache issues no network traffic. When `output_path` is omitted, the
mosaic is written to `<cache parent>/dem/<FANINSAR_DEM_NAME or dem.tif>`.


## 3. Switch products

Pass any of the fourteen names as `source`. Construction is fail-closed —
an unknown product is rejected before any network traffic.


In [4]:
for name in ("nasadem", "arcticdem-2", "nisar-glo30"):
    entry = parse_selection(name)
    print(f"{name}: provider={entry.provider}, datum={entry.vertical_datum}, "
          f"auth={entry.auth}, wired={entry.wired}")

try:
    parse_selection("srtm-x")
except Exception as err:  # unknown names fail closed, before any I/O
    print(f"rejected before any network traffic: {type(err).__name__}")

nasadem: provider=pc, datum=egm96, auth=none, wired=True
arcticdem-2: provider=aws, datum=ellipsoidal, auth=none, wired=True
nisar-glo30: provider=earthdata, datum=ellipsoidal, auth=token, wired=True
2026-08-24 21:13:20 | ERROR | faninsar.processing.geometry.dem_sources | unknown DEM product 'srtm-x'; valid products are: alos-dem, arcticdem-10, arcticdem-2, arcticdem-32, glo30, glo90, nasadem, nisar-glo30, rema-10, rema-2, rema-32, srtm-skadi, terrain-tiles


rejected before any network traffic: ValueError


Fetching a Planetary Computer product needs the `[pc]` extra because the
PC channel signs asset URLs through `planetary-computer`:

```bash
pip install "faninsar[pc]"
```

```python
dem = get_dem_manager(source="nasadem").fetch_dem(roi)
```


## 4. Switch providers manually

The provider axis is explicit: `"<product>:<provider>"`. v1 wires one
default channel per product plus three alternative identities.


In [5]:
for selection in ("nasadem:earthdata", "alos-dem:jaxa-ftp"):
    entry = parse_selection(selection)
    print(f"{selection}: product={entry.product}, provider={entry.provider}, auth={entry.auth}")

try:
    parse_selection("glo30:pc")  # not a wired pair
except Exception as err:
    print(f"unknown product/provider pair rejected: {type(err).__name__}")

nasadem:earthdata: product=nasadem, provider=earthdata, auth=token
alos-dem:jaxa-ftp: product=alos-dem, provider=jaxa-ftp, auth=none
2026-08-24 21:13:20 | ERROR | faninsar.processing.geometry.dem_sources | provider 'pc' is not wired for product 'glo30' (registered but unavailable in this release)


unknown product/provider pair rejected: ValueError


Provider prerequisites:

| Provider | Setup |
|---|---|
| `aws` | nothing — anonymous S3/HTTPS |
| `pc` | `pip install "faninsar[pc]"`; anonymous by default |
| `earthdata` | free Earthdata Login; `~/.netrc` or `EARTHDATA_TOKEN` |
| `jaxa-ftp` | nothing — anonymous FTP with multi-block zips |


## 5. What `auto` does (and does not do)

`auto` = GLO-30 with a **same-family rescue**: when a GLO-30 cell is
withheld (for example the Caucasus cells around N38-41/E045-046 return
404), only the missing cells are refetched from GLO-90 **on the same AWS
channel** and resampled back to 30 m. It never switches provider, never
switches product family, and cannot be combined with an explicit provider
(`"auto:pc"` is rejected loudly).


In [6]:
auto_mgr = get_dem_manager(source="auto")
print("auto resolves to:", auto_mgr.source_entry.name)
print("datum:", auto_mgr.vertical_datum)

try:
    get_dem_manager(source="auto:pc")  # auto never carries a provider
except Exception as err:
    print(f"auto with an explicit provider is rejected: {type(err).__name__}")

auto resolves to: auto
datum: egm2008
2026-08-24 21:13:20 | ERROR | faninsar.processing.geometry.dem_manager | invalid DEM source 'auto:pc': 'auto' accepts no provider override; select the underlying product directly (e.g. 'glo90')


auto with an explicit provider is rejected: ValueError


## 6. Automatic DEM in the pipeline

With `FANINSAR_DEM_CACHE_DIR` set, `run_pair(..., dem=None)` builds the
DEM automatically through one shared entry point,
`resolve_auto_dem(bounds, output_dir, geoid_correction, dem_source)`,
which applies the datum-aware wrap rule (orthometric sources get a geoid
adjustment; ellipsoidal sources never do) and returns a ready-to-use
sampler that the pipeline pins onto the compute device.


In [7]:
# The exact manager the pipeline would use:
pipeline_mgr = get_dem_manager()
print("pipeline selection:", pipeline_mgr.source_entry.name)

import inspect
from faninsar.processing.pipeline.production import resolve_auto_dem

print("\nresolve_auto_dem" + str(inspect.signature(resolve_auto_dem)))

pipeline selection: glo30

resolve_auto_dem(bounds: 'tuple[float, float, float, float]', *, output_dir: 'str | Path', geoid_correction: 'bool' = True, dem_source: 'str | None' = None, output_name: 'str | None' = None) -> 'DEMSampler'


The CLI exposes the same grammar through `--dem-source` (accepted values
include `auto`); unknown names are rejected in a pre-gate before any
pipeline work starts:

```bash
faninsar frame REF SEC --roi ... --dem auto-dem.tif --dem-source nasadem
```


## 7. Outage handling and environment variables

A failing provider raises `DEMProviderUnavailableError` — no hidden
cross-provider fallback. The exception carries structured fields so
callers can decide what to do next:


In [8]:
from faninsar.processing.geometry.dem_manager import DEMProviderUnavailableError

err = DEMProviderUnavailableError(
    "aws is unreachable for glo30",
    product="glo30",
    provider="aws",
    host="copernicus-dem-30m.s3.amazonaws.com",
    failure_class="upstream-outage",
    attempts=3,
    alternatives={"nasadem:earthdata": "full-quality granules via Earthdata"},
)
print(f"{err.failure_class} on {err.product}@{err.provider} ({err.host}) after {err.attempts} attempts")
print("manual escape hatches:", err.alternatives)

upstream-outage on glo30@aws (copernicus-dem-30m.s3.amazonaws.com) after 3 attempts
manual escape hatches: {'nasadem:earthdata': 'full-quality granules via Earthdata'}


`alternatives` lists the **manual** escape hatches; nothing switches on
its own, and unwired alternatives are excluded by design.

Environment variables (an explicit argument always wins over the
environment; the environment always wins over the built-in default):

| Variable | Meaning | Default |
|---|---|---|
| `FANINSAR_DEM_CACHE_DIR` | tile cache root (required for `get_dem_manager`) | — |
| `FANINSAR_DEM_SOURCE` | default selection | `glo30` |
| `FANINSAR_DEM_NAME` | mosaic file name | `dem.tif` |
| `FANINSAR_DEM_SOURCE_URL` | https base-URL override (mirror escape hatch) | provider default |
| `EARTHDATA_TOKEN` | Earthdata bearer token alternative to `~/.netrc` | — |


## Exercise

Pick a ROI and a product, then answer:

1. Which tiles cover it, and how many are already cached?
2. What is the product's vertical datum, and would `resolve_auto_dem`
   wrap it with a geoid correction?
3. Which provider alternatives exist for that product?


In [9]:
roi2 = BoundingBox(100.0, 39.0, 101.0, 40.0)
mgr2 = DEMManager(cache_dir, source="glo90")
tiles2 = mgr2.required_tiles(roi2)
cached = [t for t in tiles2 if t.cache_path.is_file()]
print(f"glo90 over the ROI: {len(tiles2)} tiles, {len(cached)} already cached")
print("datum:", mgr2.vertical_datum)

glo90 over the ROI: 4 tiles, 0 already cached
datum: egm2008


## Pitfalls and extensions

- **Bounds order.** All bounds are `(min_lon, min_lat, max_lon, max_lat)`
  in EPSG:4326 — longitude first.
- **Datum mixing.** Never feed an ellipsoidal DEM (`nisar-glo30`,
  `arcticdem-*`, `rema-*`) into code that unconditionally applies an
  EGM96/EGM2008 geoid correction; use `resolve_auto_dem` or check
  `manager.vertical_datum` first.
- **No silent failover.** A dead provider raises
  `DEMProviderUnavailableError`; switching is a caller decision informed
  by `err.alternatives`.
- **Provider auth.** `earthdata` needs `~/.netrc` or `EARTHDATA_TOKEN`;
  `pc` products need the `[pc]` extra installed.
- **Performance knobs.** `DEMManager(cache_dir, source=...,
  max_workers=8, chunked_threshold=4)` controls the parallel transport:
  raise `max_workers` on fat links; below `chunked_threshold` missing
  tiles the engine prefers ranged-chunk download of big artifacts.
- **Mirror override.** `base_url=` (or `FANINSAR_DEM_SOURCE_URL`) points
  the selected source at a different https mirror; rejected for shapes
  without a base URL (FTP, Earthdata).
